<a href="https://colab.research.google.com/github/SIRLLON/PB/blob/main/AT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assessment de Bloco (AT)

**Aluno:** Sirllon  
**Disciplina:** Projeto de Bloco  
**Plataforma:** Google Colab Notebook

Este notebook apresenta a consolidação final do projeto, unindo todos os tópicos práticos estudados e implementados do TP1 ao TP5. O conteúdo engloba algoritmos de ordenação, estruturas de dados lineares e não lineares, busca de caminhos mínimos, redes de computadores, programação assíncrona e paralelismo utilizando Python nativo e `ThreadPoolExecutor`.

# Parte 1: Ordenação e Estruturas Lineares (TP1)


## Setup: Inicialização dos Dados
Gerar a listagem de caminhos de arquivos simulados em memória para uso nos experimentos.


In [ ]:
import random

# Configurar semente aleatória para reprodutibilidade
random.seed(42)
n_arquivos = 10000
extensoes = ['.txt', '.py', '.pdf', '.png', '.docx', '.csv', '.json', '.zip']
pastas = ['src', 'bin', 'docs', 'images', 'data', 'config', 'logs', 'temp']

# Gerar caminhos em memória diretamente
lista_original = [
    f"/{random.choice(pastas)}/arquivo_{random.randint(100, 99999)}{random.choice(extensoes)}"
    for _ in range(n_arquivos)
]

print(f"Gerada lista_original em memória com {len(lista_original)} elementos.")


## Algoritmos de Ordenação
Implementar os algoritmos de ordenação Bubble Sort, Selection Sort e Insertion Sort. Realizar medições de tempo de execução com tamanhos de entrada progressivos ($N$ de 500 a 3000 itens) para analisar e comparar o desempenho das curvas de tempo.


In [ ]:
# Bubble Sort
def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr

# Selection Sort
def selection_sort(arr):
    n = len(arr)
    for i in range(n):
        min_idx = i
        for j in range(i+1, n):
            if arr[j] < arr[min_idx]:
                min_idx = j
        arr[i], arr[min_idx] = arr[min_idx], arr[i]
    return arr

# Insertion Sort
def insertion_sort(arr):
    for i in range(1, len(arr)):
        chave = arr[i]
        j = i-1
        while j >= 0 and chave < arr[j]:
            arr[j+1] = arr[j]
            j -= 1
        arr[j+1] = chave
    return arr


### Medição do Tempo de Execução
Medir o tempo de execução de cada algoritmo para diferentes tamanhos de entrada $N$ e armazenar os tempos para análise gráfica.


In [ ]:
# Garante que lista_original existe caso a parte 1 tenha sido removida
if 'lista_original' not in globals() and 'lista_original' not in locals():
    import random
    random.seed(42)
    extensoes = ['.txt', '.py', '.pdf', '.png', '.docx', '.csv', '.json', '.zip']
    pastas = ['src', 'bin', 'docs', 'images', 'data', 'config', 'logs', 'temp']
    lista_original = [
        f"/{random.choice(pastas)}/arquivo_{random.randint(100, 99999)}{random.choice(extensoes)}"
        for _ in range(10000)
    ]

import time

# Tamanhos dos subconjuntos
tamanhos = [500, 1000, 1500, 2000, 2500, 3000]
tempos_bubble = []
tempos_selection = []
tempos_insertion = []

# Roda testes
for t in tamanhos:
    # Prepara copias
    dados_bubble = lista_original[:t].copy()
    dados_selection = lista_original[:t].copy()
    dados_insertion = lista_original[:t].copy()

    # Mede Bubble
    t_ini = time.time()
    bubble_sort(dados_bubble)
    tempos_bubble.append(time.time() - t_ini)

    # Mede Selection
    t_ini = time.time()
    selection_sort(dados_selection)
    tempos_selection.append(time.time() - t_ini)

    # Mede Insertion
    t_ini = time.time()
    insertion_sort(dados_insertion)
    tempos_insertion.append(time.time() - t_ini)

    print(f"Tamanho N={t} concluido.")


### Comparativo Gráfico de Desempenho
Gerar gráfico de linhas com a biblioteca `matplotlib` para comparar o tempo de execução dos algoritmos em função do tamanho da entrada $N$.


In [ ]:
import matplotlib.pyplot as plt

# Desenha o grafico
plt.figure(figsize=(10, 6))
plt.plot(tamanhos, tempos_bubble, label='Bubble Sort', marker='o')
plt.plot(tamanhos, tempos_selection, label='Selection Sort', marker='s')
plt.plot(tamanhos, tempos_insertion, label='Insertion Sort', marker='^')
plt.title('Comparativo de Tempo de Ordenação')
plt.xlabel('Tamanho da Entrada (N)')
plt.ylabel('Tempo de Execução (segundos)')
plt.legend()
plt.grid(True)
plt.show()


## Estruturas de Dados
Implementar e testar o armazenamento dos 10.000 caminhos de arquivos em três estruturas de dados:
1. **Pilha (Stack)**: com política LIFO (Last-In, First-Out).
2. **Fila (Queue)**: com política FIFO (First-In, First-Out).
3. **Tabela Hash (HashTable)**: tabela customizada com tratamento de colisões por encadeamento.


In [ ]:
import tracemalloc
from collections import deque

# Pilha baseada em lista
class Pilha:
    def __init__(self):
        self.itens = []  # Inicializa lista

    def empilhar(self, item):
        self.itens.append(item)  # Adiciona elemento

    def desempilhar(self):
        return self.itens.pop()  # Remove topo

    def buscar(self, pos):
        return self.itens[pos]  # Acessa indice

    def tamanho(self):
        return len(self.itens)  # Retorna tamanho

# Fila baseada em deque
class Fila:
    def __init__(self):
        self.itens = deque()  # Inicializa deque

    def enfileirar(self, item):
        self.itens.append(item)  # Adiciona no fim

    def desenfileirar(self):
        return self.itens.popleft()  # Remove do inicio

    def buscar(self, pos):
        return self.itens[pos]  # Acessa indice

    def tamanho(self):
        return len(self.itens)  # Retorna tamanho

# Tabela Hash customizada
class TabelaHash:
    def __init__(self, tam=10000):
        self.tamanho_tabela = tam
        self.tabela = [[] for _ in range(tam)]  # Inicializa buckets

    def _hash(self, chave):
        return hash(chave) % self.tamanho_tabela  # Calcula hash

    def inserir(self, chave, valor):
        h = self._hash(chave)
        for par in self.tabela[h]:
            if par[0] == chave:
                par[1] = valor  # Atualiza valor
                return
        self.tabela[h].append([chave, valor])  # Adiciona par

    def buscar_chave(self, chave):
        h = self._hash(chave)
        for par in self.tabela[h]:
            if par[0] == chave:
                return par[1]  # Retorna valor
        return None  # Chave nao encontrada

    def remover(self, chave):
        h = self._hash(chave)
        for i, par in enumerate(self.tabela[h]):
            if par[0] == chave:
                return self.tabela[h].pop(i)[1]  # Remove par
        return None  # Chave nao encontrada


### Carga das Estruturas e Medição de Desempenho
Inserir os 10.000 caminhos nas três estruturas, medindo o tempo total de carga e o pico de consumo de memória de cada uma.


In [ ]:
# Garante que lista_original existe caso a parte 1 tenha sido removida
if 'lista_original' not in globals() and 'lista_original' not in locals():
    import random
    random.seed(42)
    extensoes = ['.txt', '.py', '.pdf', '.png', '.docx', '.csv', '.json', '.zip']
    pastas = ['src', 'bin', 'docs', 'images', 'data', 'config', 'logs', 'temp']
    lista_original = [
        f"/{random.choice(pastas)}/arquivo_{random.randint(100, 99999)}{random.choice(extensoes)}"
        for _ in range(10000)
    ]

# Inicializa estruturas
pilha = Pilha()
fila = Fila()
hash_table = TabelaHash()

# Mede Pilha
tracemalloc.start()
t_ini = time.time()
for i, arq in enumerate(lista_original):
    pilha.empilhar(arq)
tempo_pilha = time.time() - t_ini
memoria_pilha = tracemalloc.get_traced_memory()[1] # Pico de memoria
tracemalloc.stop()

# Mede Fila
tracemalloc.start()
t_ini = time.time()
for i, arq in enumerate(lista_original):
    fila.enfileirar(arq)
tempo_fila = time.time() - t_ini
memoria_fila = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

# Mede Hash Table
tracemalloc.start()
t_ini = time.time()
for i, arq in enumerate(lista_original):
    hash_table.inserir(i, arq)
tempo_hash = time.time() - t_ini
memoria_hash = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

print(f"Pilha - Tempo: {tempo_pilha:.5f}s | Memória: {memoria_pilha / 1024:.2f} KB")
print(f"Fila  - Tempo: {tempo_fila:.5f}s | Memória: {memoria_fila / 1024:.2f} KB")
print(f"Hash  - Tempo: {tempo_hash:.5f}s | Memória: {memoria_hash / 1024:.2f} KB")


### Busca de Elementos em Posições Específicas
Buscar elementos em posições predefinidas (índices 0, 99, 999, 4999 e 9999) em cada estrutura de dados e medir o tempo de busca acumulado.


In [ ]:
posicoes = [0, 99, 999, 4999, 9999]  # Indices correspondentes

print("=== BUSCAS NA PILHA ===")
t_ini = time.time()
for pos in posicoes:
    arq = pilha.buscar(pos)
    print(f"Posição {pos+1}: {arq}")
tempo_busca_pilha = time.time() - t_ini
print(f"Tempo de busca total na Pilha: {tempo_busca_pilha * 1000:.6f} ms\n")

print("=== BUSCAS NA FILA ===")
t_ini = time.time()
for pos in posicoes:
    arq = fila.buscar(pos)
    print(f"Posição {pos+1}: {arq}")
tempo_busca_fila = time.time() - t_ini
print(f"Tempo de busca total na Fila: {tempo_busca_fila * 1000:.6f} ms\n")

print("=== BUSCAS NA TABELA HASH ===")
t_ini = time.time()
for pos in posicoes:
    arq = hash_table.buscar_chave(pos)
    print(f"Posição {pos+1}: {arq}")
tempo_busca_hash = time.time() - t_ini
print(f"Tempo de busca total na Tabela Hash: {tempo_busca_hash * 1000:.6f} ms")


### Adição e Remoção de Elementos
Inserir e remover um elemento adicional nas três estruturas de dados, mensurando o tempo de execução e o consumo de memória das operações combinadas.


In [ ]:
novo_elemento = "/temp/novo_arquivo_adicionado.py"

# Pilha (Adicionar/Remover)
tracemalloc.start()
t_ini = time.time()
pilha.empilhar(novo_elemento)  # Insere elemento
removido_pilha = pilha.desempilhar()  # Remove elemento
tempo_op_pilha = time.time() - t_ini
memoria_op_pilha = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()
print(f"Pilha (Add/Remove) - Tempo: {tempo_op_pilha * 1000:.6f} ms | Memória: {memoria_op_pilha} bytes")

# Fila (Adicionar/Remover)
tracemalloc.start()
t_ini = time.time()
fila.enfileirar(novo_elemento)  # Insere elemento
removido_fila = fila.desenfileirar()  # Remove elemento
tempo_op_fila = time.time() - t_ini
memoria_op_fila = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()
print(f"Fila (Add/Remove)  - Tempo: {tempo_op_fila * 1000:.6f} ms | Memória: {memoria_op_fila} bytes")

# Hash (Adicionar/Remover)
tracemalloc.start()
t_ini = time.time()
hash_table.inserir(10000, novo_elemento)  # Insere elemento
removido_hash = hash_table.remover(10000)  # Remove elemento
tempo_op_hash = time.time() - t_ini
memoria_op_hash = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()
print(f"Hash (Add/Remove)  - Tempo: {tempo_op_hash * 1000:.6f} ms | Memória: {memoria_op_hash} bytes")


## Parte 3: Análise e Relatório Teórico

### 1. Tempo de Execução dos Algoritmos de Ordenação (Big O)

A tabela abaixo mostra o tempo teórico que cada algoritmo leva para rodar:

| Algoritmo | Melhor Caso | Caso Médio | Pior Caso | Memória Extra Utilizada |
| :--- | :---: | :---: | :---: | :---: |
| **Bubble Sort** | $O(N)$ | $O(N^2)$ | $O(N^2)$ | $O(1)$ |
| **Selection Sort** | $O(N^2)$ | $O(N^2)$ | $O(N^2)$ | $O(1)$ |
| **Insertion Sort** | $O(N)$ | $O(N^2)$ | $O(N^2)$ | $O(1)$ |

#### Explicação Simples do Funcionamento:
- **Bubble Sort**: Comparar repetidamente os elementos vizinhos (lado a lado) e trocá-los de lugar se estiverem na ordem errada. O maior elemento é levado para o final a cada leitura completa da lista. Mesmo na melhor situação, sem uma verificação para parar mais cedo, o algoritmo percorre a lista inteira todas as vezes.
- **Selection Sort**: Dividir a lista em duas partes: a organizada e a não organizada. Em cada etapa, procurar o menor elemento da parte não organizada e trocá-lo de lugar com o primeiro elemento dessa parte. Como sempre precisa olhar todos os elementos restantes para encontrar o menor valor, o tempo de execução aumenta de forma quadrática, não importa como a lista começava.
- **Insertion Sort**: Montar a lista organizada passo a passo, pegando um elemento de cada vez e colocando-o na posição certa no grupo de elementos que já foram organizados. Funciona muito bem para listas pequenas ou que já estão quase na ordem correta.

#### Análise Prática com base no Gráfico:
De acordo com os resultados do gráfico:
- O **Bubble Sort** teve o pior tempo de execução, demorando muito mais à medida que o tamanho da lista ($N$) cresceu.
- O **Selection Sort** e o **Insertion Sort** terminaram bem mais rápido que o Bubble Sort, mas a curva de tempo também mostra que eles demoram mais conforme a lista aumenta.
- O **Insertion Sort** funciona melhor na prática do que os outros dois porque ele consegue parar a busca interna assim que encontra o lugar certo do elemento, o que corta o número de comparações pela metade comparado ao Selection Sort.

### 2. Análise Teórica e Prática das Estruturas de Dados

#### Tabela Comparativa de Resultados Reais:
*(Dados obtidos a partir da última execução do notebook)*

| Estrutura | Tempo de Carga (10k itens) | Memória Utilizada (KB) | Tempo de Acesso (Médio) | Adição/Remoção (Tempo) | Adição/Remoção (Memória) |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Pilha** | Rápido (~0.002s) | Baixo (~140 KB) | $O(1)$ | Instantâneo | Baixo |
| **Fila** | Rápido (~0.002s) | Baixo (~160 KB) | $O(1)$ | Instantâneo | Baixo |
| **Tabela Hash** | Moderado (~0.008s) | Médio/Alto (~800 KB) | $O(1)$ (Média) | Instantâneo | Baixo |

#### Discussão das Propriedades Teóricas e de Memória:

1. **Tempo para Carregar os Dados (Carga)**:
   - **Pilha e Fila** são extremamente rápidas para adicionar muitos elementos de uma vez. Na Pilha, a adição é instantânea na média ($O(1)$). Na Fila, a adição também possui tempo constante garantido ($O(1)$) por usar partes conectadas na memória (`deque`).
   - A **Tabela Hash** leva um pouco mais de tempo para carregar porque precisa calcular um código de busca (função hash) para cada arquivo e tratar os casos em que dois arquivos geram o mesmo código (colisões).

2. **Consumo de Memória**:
   - **Pilha** e **Fila** usam muito menos memória porque apenas guardam a sequência simples de dados, um atrás do outro.
   - A **Tabela Hash** gasta cerca de 5 a 6 vezes mais memória porque precisa reservar espaço antecipado para a estrutura de índices e gerenciar listas internas para organizar os arquivos que empataram no mesmo código de busca.

3. **Tempo para Achar Elementos (Acesso)**:
   - Para a **Pilha** e a **Fila**, encontrar um arquivo por sua posição é feito de forma imediata ($O(1)$) porque as listas do Python conseguem acessar qualquer índice direto na memória.
   - Para a **Tabela Hash**, ao usar a posição como chave de busca, o acesso também é imediato ($O(1)$) em média, pois o código de busca leva diretamente ao local do arquivo.

4. **Adicionar e Remover Elementos**:
   - **Pilha (LIFO)**: Colocar e tirar elementos no topo são imediatos ($O(1)$) porque ocorrem sempre no fim, sem precisar mexer no resto da estrutura.
   - **Fila (FIFO)**: Colocar no fim e tirar do começo também são operações imediatas ($O(1)$) usando `deque`, pois não é necessário reorganizar a fila na memória.
   - **Tabela Hash**: Inserir e remover elementos são muito rápidos no caso médio ($O(1)$). Os tempos medidos em milissegundos mostram que essas estruturas são excelentes para atualizações rápidas.


### 2. Análise Teórica e Prática das Estruturas de Dados

#### Tabela Comparativa de Resultados Reais:
*(Dados obtidos a partir da última execução do notebook)*

| Estrutura | Tempo de Carga (10k itens) | Memória de Carga (KB) | Tempo de Acesso (Médio) | Adição/Remoção (Tempo) | Adição/Remoção (Memória) |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Pilha** | Rápido (~0.002s) | Baixo (~140 KB) | $O(1)$ | Instantâneo | Baixo |
| **Fila** | Rápido (~0.002s) | Baixo (~160 KB) | $O(1)$ | Instantâneo | Baixo |
| **Tabela Hash** | Moderado (~0.008s) | Médio/Alto (~800 KB) | $O(1)$ (Média) | Instantâneo | Baixo |

#### Discussão das Propriedades Teóricas e de Memória:

1. **Tempo de Inserção (Carga)**:
   - **Pilha e Fila** são extremamente rápidas para inserção em lote. Na Pilha, o `append` é $O(1)$ amortizado. Na Fila, o `append` na estrutura `deque` também possui custo $O(1)$ constante garantido por ser baseada em blocos encadeados.
   - A **Tabela Hash** leva mais tempo para ser carregada por precisar calcular a função hash (`hash(chave)`) para cada arquivo e tratar possíveis colisões nos buckets, além de redimensionamentos dinâmicos implicitamente gerenciados.

2. **Consumo de Memória**:
   - **Pilha** e **Fila** mantêm as menores pegadas de memória pois apenas guardam referências lineares consecutivas aos dados.
   - A **Tabela Hash** consome cerca de 5 a 6 vezes mais memória devido ao vetor de buckets pré-alocado (para o endereçamento) e ao fato de que cada bucket é uma lista aninhada (com ponteiros adicionais) para gerenciar o encadeamento de colisões.

3. **Recuperação de Elementos**:
   - Para a **Pilha** e a **Fila**, a recuperação direta por índice (posição 1, 100, etc.) no Python é feita em tempo constante $O(1)$ pois as implementações internas subjacentes (lista contígua do Python e blocos encadeados do `deque`) mapeiam índices de forma muito eficiente.
   - Para a **Tabela Hash**, ao mapearmos a posição numérica como chave, o acesso é $O(1)$ constante em média, pois a chave aponta diretamente para o bucket via cálculo de hash.

4. **Adição e Remoção de Itens**:
   - **Pilha (LIFO)**: Inserção e remoção no topo (`empilhar`/`desempilhar`) são $O(1)$ porque ocorrem sempre no fim do vetor, sem necessidade de deslocamento de outros elementos.
   - **Fila (FIFO)**: Inserção no fim (`enfileirar`) e remoção no início (`desenfileirar`) operam em $O(1)$ constante usando `deque`, pois não há necessidade de reorganizar a memória.
   - **Tabela Hash**: Inserção e remoção são $O(1)$ no caso médio. O tempo registrado em milissegundos é extremamente pequeno, validando a alta eficiência dessas estruturas para operações dinâmicas.


# Parte 2: Concorrência, Recursividade e Ordenação Avançada (TP2)


### Exercício 1: Coleta de Dados com Asyncio
Executa 5 tarefas assíncronas em paralelo e agrega os resultados.

In [ ]:
import asyncio
import random

async def coletar_dados(id_tarefa):
    # Simula espera de rede
    await asyncio.sleep(random.uniform(0.1, 0.5))
    return f"Dados {id_tarefa}"

async def main_ex1():
    # Cria lista de tarefas
    tarefas = [coletar_dados(i) for i in range(1, 6)]
    # Aguarda todas as tarefas
    resultados = await asyncio.gather(*tarefas)
    print("Resultados agregados:", resultados)

await main_ex1()

### Exercício 2: asyncio.to_thread()
Mede a diferença entre chamar uma função bloqueante diretamente ou encapsulada em uma thread do event loop.

In [ ]:
import asyncio
import time

def funcao_custosa():
    # Simula carga CPU sincrona
    time.sleep(1)
    return "Finalizado"

async def main_ex2():
    # Mede tempo direto
    t_ini = time.time()
    funcao_custosa()
    print(f"Direto bloqueante: {time.time() - t_ini:.3f} s")

    # Mede com to_thread
    t_ini = time.time()
    await asyncio.to_thread(funcao_custosa)
    print(f"Com to_thread: {time.time() - t_ini:.3f} s")

await main_ex2()

### Exercício 3: Pipeline Assíncrono (E1 -> E2 -> E3)
Implementação de pipeline em fluxo que processa itens concorrentemente.

In [ ]:
import asyncio
import time

async def coleta(item_id):
    # Coleta dados assincronamente
    await asyncio.sleep(0.1)
    return f"dados_{item_id}"

def transformacao(dados):
    # Converte para maiusculas
    return dados.upper()

async def envio(dados):
    # Envia dados assincronamente
    await asyncio.sleep(0.1)
    print(f"Enviado: {dados}")

async def processar_item(item_id):
    # Fluxo do pipeline
    item = await coleta(item_id)
    trans = transformacao(item)
    await envio(trans)

async def main_ex3():
    # Processa multiplos itens
    await asyncio.gather(*(processar_item(i) for i in range(1, 4)))

await main_ex3()

### Exercício 4: Deadlock asyncio vs OpenMP (Explicação)
Um **deadlock** ocorre quando uma tarefa do `asyncio` bloqueia a thread única do *event loop* esperando que threads secundárias do `OpenMP` concluam uma região paralela. Ao mesmo tempo, se alguma thread do `OpenMP` precisar submeter um callback ou interagir com o event loop (aguardando resposta), ela ficará travada eternamente, pois a thread do event loop está bloqueada aguardando o fim do OpenMP.

### Exercício 5: Servidor TCP Eco com asyncio
Servidor de eco que recebe dados e retorna a mesma string prefixada.

In [ ]:
import asyncio

async def lidar_cliente(reader, writer):
    # Le dados enviados
    dados = await reader.read(100)
    mensagem = dados.decode()
    # Retorna o eco
    writer.write(f"ECO: {mensagem}".encode())
    await writer.drain()
    writer.close()

async def main_ex5():
    # Inicia o servidor
    servidor = await asyncio.start_server(lidar_cliente, '127.0.0.1', 8889)
    print("Servidor TCP rodando em 127.0.0.1:8889")
    async with servidor:
        await asyncio.sleep(0.2) # Mantem ativo por instantes
        servidor.close()

await main_ex5()

### Exercício 6: QuickSelect
Encontra o k-ésimo menor elemento de um vetor.

In [ ]:
def partition(arr, l, r):
    # Define pivo simples
    pivot = arr[r]
    i = l
    for j in range(l, r):
        if arr[j] <= pivot:
            arr[i], arr[j] = arr[j], arr[i]
            i += 1
    arr[i], arr[r] = arr[r], arr[i]
    return i

def quickselect(arr, l, r, k):
    # Algoritmo de selecao
    if l <= r:
        p_idx = partition(arr, l, r)
        if p_idx == k:
            return arr[k]
        elif p_idx > k:
            return quickselect(arr, l, p_idx - 1, k)
        else:
            return quickselect(arr, p_idx + 1, r, k)
    return -1

vetor = [12, 3, 5, 7, 4, 19, 26]
print("3º menor:", quickselect(vetor.copy(), 0, len(vetor)-1, 2))

### Exercício 7: QuickSelect com Histórico de Pivôs
Ajusta a seleção do pivô com base em histórico.

In [ ]:
# Memoriza ultimos pivos
historico_pivos = []

def escolhe_pivo_historico(arr, l, r, k_hist):
    # Escolhe pivo considerando historico
    p_idx = r
    if len(historico_pivos) >= k_hist:
        # Usa a mediana do historico
        p_idx = int(sum(historico_pivos[-k_hist:]) / k_hist) % (r - l + 1) + l
    historico_pivos.append(p_idx)
    return p_idx

### Exercício 8: QuickSort Mediana de Três vs Simples
Compara QuickSort simples com pivotagem de mediana.

In [ ]:
def quicksort_simples(arr):
    # QuickSort basico recursivo
    if len(arr) <= 1: return arr
    piv = arr[len(arr)//2]
    esq = [x for x in arr if x < piv]
    meio = [x for x in arr if x == piv]
    dir = [x for x in arr if x > piv]
    return quicksort_simples(esq) + meio + quicksort_simples(dir)

def mediana_de_tres(arr, l, r):
    mid = (l + r) // 2
    vals = [(arr[l], l), (arr[mid], mid), (arr[r], r)]
    vals.sort()
    return vals[1][1]

def quicksort_m3(arr, l, r):
    # QuickSort mediana de 3
    if l < r:
        p_idx = mediana_de_tres(arr, l, r)
        arr[p_idx], arr[r] = arr[r], arr[p_idx]
        p = partition(arr, l, r)
        quicksort_m3(arr, l, p - 1)
        quicksort_m3(arr, p + 1, r)

### Exercício 9: QuickSort com Dois Pivôs (Explicação)
O **Dual-Pivot QuickSort** divide o vetor em três partes usando dois pivôs $P_1$ e $P_2$ (onde $P_1 \le P_2$). Ele classifica os elementos em: menores que $P_1$, entre $P_1$ e $P_2$, e maiores que $P_2$.  
* **Vantagens**: Reduz o número de acessos e trocas de memória na prática, otimizando o cache.  
* **Desvantagens**: Particionamento e controle de índices significativamente mais complexos.

### Exercício 10: QuickSort Híbrido com Insertion Sort
Evita recursão profunda para pequenos subvetores (< 10 elementos).

In [ ]:
def insertion_sort(arr, l, r):
    # Ordenacao por insercao
    for i in range(l + 1, r + 1):
        key = arr[i]
        j = i - 1
        while j >= l and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key

def quicksort_hibrido(arr, l, r):
    # QuickSort com Insertion Sort
    if l < r:
        if r - l < 10:
            insertion_sort(arr, l, r)
        else:
            p = partition(arr, l, r)
            quicksort_hibrido(arr, l, p - 1)
            quicksort_hibrido(arr, p + 1, r)

### Exercício 11: Operações Básicas de Lista Encadeada
Implementação de nó e lista encadeada simples.

In [ ]:
class Node:
    def __init__(self, val):
        self.val = val
        self.next = None

class ListaEncadeada:
    def __init__(self):
        self.head = None

    def inserir_inicio(self, val):
        # Adiciona no inicio
        novo = Node(val)
        novo.next = self.head
        self.head = novo

    def inserir_fim(self, val):
        # Adiciona no fim
        novo = Node(val)
        if not self.head:
            self.head = novo
            return
        curr = self.head
        while curr.next: curr = curr.next
        curr.next = novo

    def remover(self, val):
        # Remove um elemento
        curr = self.head
        prev = None
        while curr:
            if curr.val == val:
                if prev: prev.next = curr.next
                else: self.head = curr.next
                return True
            prev = curr
            curr = curr.next
        return False

    def buscar(self, val):
        # Busca por valor
        curr = self.head
        while curr:
            if curr.val == val: return True
            curr = curr.next
        return False

### Exercício 12: Lista com Blocos de B Elementos (Explicação)
Uma **lista de blocos contíguos de tamanho B** (Unrolled Linked List) armazena múltiplos elementos contíguos em cada nó.
* **Vantagens**: Excelente localidade de cache e redução do número de ponteiros necessários na memória.
* **Desvantagens**: Inserções/remoções internas requerem deslocamento de dados complexos ou cisão (splitting) de nós.

### Exercício 13: Lista Duplamente Encadeada Ordenada
Mantém elementos em ordem com ponteiros bidirecionais.

In [ ]:
class DoublyNode:
    def __init__(self, val):
        self.val = val
        self.next = None
        self.prev = None

class DoublyLinkedListSorted:
    def __init__(self):
        self.head = None

    def inserir(self, val):
        # Insere mantendo a ordenacao
        novo = DoublyNode(val)
        if not self.head:
            self.head = novo
            return novo
        curr = self.head
        while curr and curr.val < val:
            if not curr.next:
                curr.next = novo
                novo.prev = curr
                return novo
            curr = curr.next
        if curr == self.head:
            novo.next = self.head
            self.head.prev = novo
            self.head = novo
        else:
            p = curr.prev
            p.next = novo
            novo.prev = p
            novo.next = curr
            curr.prev = novo
        return novo

    def remover_nodo(self, node):
        # Remove nodo por referencia
        if not node: return
        if node == self.head:
            self.head = node.next
        if node.next: node.next.prev = node.prev
        if node.prev: node.prev.next = node.next

### Exercício 14: Lista Circular
Estrutura encadeada onde o último nó aponta para o primeiro.
**Casos de uso reais**:
1. Escalonamento de Processos Round-Robin em Sistemas Operacionais.
2. Gerenciamento de turnos em jogos de tabuleiro multiplayer digitais.

In [ ]:
class CircularList:
    def __init__(self):
        self.head = None

    def inserir(self, val):
        # Insere na lista circular
        novo = Node(val)
        if not self.head:
            self.head = novo
            novo.next = novo
            return
        curr = self.head
        while curr.next != self.head:
            curr = curr.next
        curr.next = novo
        novo.next = self.head

### Exercício 15: Tabela Hash com Lista Encadeada para Colisão
Armazena pares chave-valor resolvendo colisões com encadeamento.

In [ ]:
class HashNode:
    def __init__(self, key, val):
        self.key = key
        self.val = val
        self.next = None

class HashTableChaining:
    def __init__(self, size=10):
        self.size = size
        self.buckets = [None] * size

    def _hash(self, key):
        return hash(key) % self.size

    def inserir(self, key, val):
        # Insere par chave valor
        idx = self._hash(key)
        novo = HashNode(key, val)
        if not self.buckets[idx]:
            self.buckets[idx] = novo
        else:
            curr = self.buckets[idx]
            while curr:
                if curr.key == key:
                    curr.val = val
                    return
                if not curr.next:
                    curr.next = novo
                    return
                curr = curr.next

    def buscar(self, key):
        # Busca por chave
        idx = self._hash(key)
        curr = self.buckets[idx]
        while curr:
            if curr.key == key: return curr.val
            curr = curr.next
        return None

### Exercício 16: Soma Recursiva de Vetor
Soma os n primeiros elementos do vetor.

In [ ]:
def soma_recursiva(arr, n):
    # Soma recursiva de vetor
    if n <= 0: return 0
    return arr[n-1] + soma_recursiva(arr, n-1)

print("Soma [1,2,3,4]:", soma_recursiva([1, 2, 3, 4], 4))

### Exercício 17: Busca Binária Recursiva
Encontra elemento no vetor ordenado recursivamente.

In [ ]:
def busca_binaria_rec(arr, x, l, r):
    # Busca binaria recursiva
    if l > r: return -1
    mid = (l + r) // 2
    if arr[mid] == x: return mid
    elif arr[mid] > x: return busca_binaria_rec(arr, x, l, mid-1)
    else: return busca_binaria_rec(arr, x, mid+1, r)

print("Busca 5 em [1,3,5,7]:", busca_binaria_rec([1, 3, 5, 7], 5, 0, 3))

### Exercício 18: Busca Binária Recursiva Híbrida (Iterativa para Profundidade > D)
Troca para modo iterativo se a pilha de recursão atingir um limite D.

In [ ]:
def busca_binaria_hibrida(arr, x, l, r, prof, D):
    # Troca de modo se prof >= D
    if prof >= D:
        # Roda busca iterativa
        while l <= r:
            mid = (l + r) // 2
            if arr[mid] == x: return mid
            elif arr[mid] > x: r = mid - 1
            else: l = mid + 1
        return -1
    else:
        if l > r: return -1
        mid = (l + r) // 2
        if arr[mid] == x: return mid
        elif arr[mid] > x: return busca_binaria_hibrida(arr, x, l, mid-1, prof+1, D)
        else: return busca_binaria_hibrida(arr, x, mid+1, r, prof+1, D)

### Exercício 19: Torre de Hanoi
Resolve recursivamente a transferência de discos.

In [ ]:
def hanoi(n, orig, dest, aux):
    # Resolve torre de hanoi
    if n == 1:
        print(f"Mover disco 1 de {orig} para {dest}")
        return
    hanoi(n-1, orig, aux, dest)
    print(f"Mover disco {n} de {orig} para {dest}")
    hanoi(n-1, aux, dest, orig)

hanoi(3, 'A', 'C', 'B')

### Exercício 20: Contagem de Dígitos Recursiva
Calcula quantos dígitos possui um número positivo.

In [ ]:
def contar_digitos(n):
    # Conta digitos recursivamente
    if n < 10: return 1
    return 1 + contar_digitos(n // 10)

print("Digitos de 1459:", contar_digitos(1459))

## Programação Concorrente em Python (Exercícios 21 ao 25)

Nesta seção, implementamos os exercícios de paralelismo usando a linguagem Python com a biblioteca `ThreadPoolExecutor` do módulo padrão `concurrent.futures`, em substituição ao OpenMP em C++. Isso simplifica a execução e teste diretamente no ambiente de execução do Colab.

### Exercício 21: Soma de Elementos com Parallel For + Reduction
Implementar a soma dos elementos de um vetor dividindo a carga de trabalho entre threads. Os resultados locais de cada thread são reduzidos (somados) na thread principal.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def soma_paralela(vetor, num_threads=4):
    n = len(vetor)
    # Divide o trabalho igualmente em blocos para cada thread
    tamanho_bloco = (n + num_threads - 1) // num_threads

    def somar_bloco(inicio):
        fim = min(inicio + tamanho_bloco, n)
        soma_local = 0
        for i in range(inicio, fim):
            soma_local += vetor[i]
        return soma_local

    inicios = [i * tamanho_bloco for i in range(num_threads)]

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        # Mapeia a execução da função somar_bloco em paralelo
        resultados = list(executor.map(somar_bloco, inicios))

    # Redução: combina os resultados parciais
    return sum(resultados)

# Testar a implementação
vetor_teste = [1] * 1000000
t_ini = time.time()
resultado = soma_paralela(vetor_teste, num_threads=4)
print(f"Soma paralela (4 threads): {resultado} | Tempo: {time.time() - t_ini:.4f} s")


### Exercício 22: Distribuição de Blocos da Lista por Threads
Implementar uma abordagem em que cada thread processa um conjunto de blocos da lista (distribuição em formato round-robin de blocos de dados entre threads).

In [ ]:
def processar_blocos_round_robin(lista_de_blocos, num_threads=4):
    def processar_grupo_de_blocos(id_thread):
        soma_local = 0
        # Cada thread consome blocos com passo fixo (round-robin) para balanceamento
        for i in range(id_thread, len(lista_de_blocos), num_threads):
            bloco = lista_de_blocos[i]
            soma_local += sum(bloco)
        return soma_local

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        resultados = list(executor.map(processar_grupo_de_blocos, range(num_threads)))

    return sum(resultados)

# Testar a implementação com blocos de dados
blocos_teste = [[i] * 1000 for i in range(50)]
resultado = processar_blocos_round_robin(blocos_teste, num_threads=4)
print(f"Soma com distribuição de blocos (round-robin): {resultado}")


### Exercício 23: Balanceamento de Carga com Prefix Sums
Implementar política de balanceamento de carga baseada na soma de prefixos (prefix sums) para distribuir blocos de tamanhos variados uniformemente entre as threads.

In [ ]:
import bisect
import random

def prefix_sums_balance(lista_de_blocos, num_threads=4):
    tamanhos = [len(bloco) for bloco in lista_de_blocos]

    # Calcular soma de prefixos (prefix sums) do esforço de trabalho
    prefix_sums = [0]
    soma_acumulada = 0
    for tam in tamanhos:
        soma_acumulada += tam
        prefix_sums.append(soma_acumulada)

    total_trabalho = prefix_sums[-1]
    trabalho_por_thread = total_trabalho / num_threads

    # Encontrar as divisões dos blocos usando busca binária (bisect)
    divisoes = [0]
    for i in range(1, num_threads):
        alvo = i * trabalho_por_thread
        idx = bisect.bisect_left(prefix_sums, alvo)
        # Ajustar para o bloco mais próximo da divisão ideal de carga
        if idx > 1 and abs(prefix_sums[idx-1] - alvo) < abs(prefix_sums[idx] - alvo):
            idx = idx - 1
        divisoes.append(idx)
    divisoes.append(len(lista_de_blocos))

    # Função para cada thread processar seu intervalo de blocos balanceado
    def processar_intervalo(id_thread):
        inicio_bloco = divisoes[id_thread]
        fim_bloco = divisoes[id_thread + 1]
        soma_local = 0
        for b_idx in range(inicio_bloco, fim_bloco):
            soma_local += sum(lista_de_blocos[b_idx])
        return soma_local

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        resultados = list(executor.map(processar_intervalo, range(num_threads)))

    return sum(resultados)

# Testar a implementação com blocos de tamanhos muito variados
random.seed(42)
blocos_variados = [[j] * random.randint(10, 5000) for j in range(100)]
resultado = prefix_sums_balance(blocos_variados, num_threads=4)
print(f"Soma balanceada por prefix sums: {resultado}")


### Exercício 24: Busca Paralela em Vetor Ordenado
Implementar busca binária paralela, ativando o paralelismo apenas quando o tamanho do vetor é suficientemente grande para mitigar o overhead de criação de threads.

In [ ]:
def busca_paralela_ordenada(arr, alvo, num_threads=4):
    n = len(arr)
    limite = 100000  # Só paraleliza se for maior que 100.000 elementos

    def busca_binaria(inicio, fim):
        esq, dir = inicio, fim
        while esq <= dir:
            meio = (esq + dir) // 2
            if arr[meio] == alvo:
                return meio
            elif arr[meio] < alvo:
                esq = meio + 1
            else:
                dir = meio - 1
        return -1

    if n <= limite:
        # Execução sequencial direta
        print("Modo de busca sequencial ativo.")
        return busca_binaria(0, n - 1)

    # Execução paralela: divide o vetor ordenado em subvetores
    print("Modo de busca paralela ativo.")
    tamanho_bloco = (n + num_threads - 1) // num_threads

    def buscar_na_thread(id_thread):
        inicio = id_thread * tamanho_bloco
        fim = min(inicio + tamanho_bloco - 1, n - 1)
        # Otimização: busca apenas se o alvo estiver dentro da faixa desse bloco
        if arr[inicio] <= alvo <= arr[fim]:
            return busca_binaria(inicio, fim)
        return -1

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        resultados = list(executor.map(buscar_na_thread, range(num_threads)))

    for res in resultados:
        if res != -1:
            return res
    return -1

# Testar a busca
vetor_ordenado = list(range(200000))
pos_alvo = busca_paralela_ordenada(vetor_ordenado, 123456, num_threads=4)
print(f"Elemento encontrado na posição: {pos_alvo}")


### Exercício 25: Multiplicação Paralela de Matrizes
Implementar multiplicação de matrizes de forma paralela, mapeando a computação de linhas independentes da matriz resultante para diferentes threads.

In [ ]:
def multiplicacao_matrizes_paralela(A, B, num_threads=4):
    N = len(A)
    # Matriz resultante inicializada com zeros
    C = [[0] * N for _ in range(N)]

    # Função que calcula a multiplicação para um subconjunto de linhas
    def multiplicar_bloco_linhas(inicio_linha):
        linhas_por_thread = (N + num_threads - 1) // num_threads
        fim_linha = min(inicio_linha + linhas_por_thread, N)
        for i in range(inicio_linha, fim_linha):
            for j in range(N):
                soma = 0
                for k in range(N):
                    soma += A[i][k] * B[k][j]
                C[i][j] = soma

    # Distribuir as linhas iniciais para cada thread
    linhas_por_thread = (N + num_threads - 1) // num_threads
    inicios = [i * linhas_por_thread for i in range(num_threads)]

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        list(executor.map(multiplicar_bloco_linhas, inicios))

    return C

# Testar com duas matrizes 100 x 100
matriz_A = [[1] * 100 for _ in range(100)]
matriz_B = [[2] * 100 for _ in range(100)]
t_ini = time.time()
matriz_C = multiplicacao_matrizes_paralela(matriz_A, matriz_B, num_threads=4)
print(f"Multiplicação paralela (100x100) concluída | Tempo: {time.time() - t_ini:.4f} s | Valor C[0][0]: {matriz_C[0][0]}")


# Parte 3: Árvores de Busca Binária (BST), Árvores Trie e Concorrência (TP3)


### Grupo 1: Árvore Binária de Busca (BST)
Implementação de inserção, busca, remoção, cálculo de altura e testes de tempo de execução.

In [ ]:
import time
import random

class BSTNode:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None

class BST:
    def __init__(self):
        self.root = None

    def inserir(self, key):
        # Insere valor na BST
        self.root = self._inserir(self.root, key)

    def _inserir(self, root, key):
        if not root: return BSTNode(key)
        if key < root.key: root.left = self._inserir(root.left, key)
        else: root.right = self._inserir(root.right, key)
        return root

    def buscar(self, key):
        # Busca elemento na arvore
        return self._buscar(self.root, key)

    def _buscar(self, root, key):
        if not root or root.key == key: return root
        if key < root.key: return self._buscar(root.left, key)
        return self._buscar(root.right, key)

    def remover(self, key):
        # Remove valor da BST
        self.root = self._remover(self.root, key)

    def _remover(self, root, key):
        if not root: return root
        if key < root.key: root.left = self._remover(root.left, key)
        elif key > root.key: root.right = self._remover(root.right, key)
        else:
            if not root.left: return root.right
            if not root.right: return root.left
            temp = self._min_val_node(root.right)
            root.key = temp.key
            root.right = self._remover(root.right, temp.key)
        return root

    def _min_val_node(self, node):
        curr = node
        while curr.left: curr = curr.left
        return curr

    def altura(self):
        # Retorna altura da arvore
        return self._altura(self.root)

    def _altura(self, root):
        if not root: return -1
        return 1 + max(self._altura(root.left), self._altura(root.right))

In [ ]:
# Medicoes para N = 100, 1000 e 10000
for n in [100, 1000, 10000]:
    bst = BST()
    dados = [random.randint(1, 100000) for _ in range(n)]

    # Mede insercoes
    t_ini = time.time()
    for d in dados: bst.inserir(d)
    t_ins = time.time() - t_ini

    # Mede buscas
    t_ini = time.time()
    for d in dados[:50]: bst.buscar(d)
    t_bus = time.time() - t_ini

    print(f"N={n:5} | Inserção: {t_ins:.6f}s | Busca (50 itens): {t_bus:.6f}s")

### Grupo 2 & 3: Computação Paralela com OpenMP e BST
Demonstrações de soma de vetores, produto escalar paralelo e maior elemento em BST.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

# Ex 6: Soma de vetores de forma paralela
def soma_vetores_paralela(a, b, num_threads=4):
    n = len(a)
    c = [0] * n
    tamanho_bloco = (n + num_threads - 1) // num_threads

    def somar_bloco(inicio):
        fim = min(inicio + tamanho_bloco, n)
        for i in range(inicio, fim):
            c[i] = a[i] + b[i]

    inicios = [i * tamanho_bloco for i in range(num_threads)]
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        list(executor.map(somar_bloco, inicios))
    return c

# Ex 7: Produto escalar com redução paralelo
def produto_escalar_paralelo(a, b, num_threads=4):
    n = len(a)
    tamanho_bloco = (n + num_threads - 1) // num_threads

    def multiplicar_bloco(inicio):
        fim = min(inicio + tamanho_bloco, n)
        soma_local = 0.0
        for i in range(inicio, fim):
            soma_local += a[i] * b[i]
        return soma_local

    inicios = [i * tamanho_bloco for i in range(num_threads)]
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        resultados = list(executor.map(multiplicar_bloco, inicios))
    return sum(resultados)

# Ex 10 e 11: Contagem de primos com paralelismo
def eh_primo(n):
    if n <= 1: return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0: return False
    return True

def contar_primos_paralelo(N, num_threads=4):
    elementos = list(range(2, N + 1))
    total_elem = len(elementos)
    chunk_size = (total_elem + num_threads - 1) // num_threads

    def contar_no_chunk(chunk_idx):
        start = chunk_idx * chunk_size
        end = min(start + chunk_size, total_elem)
        count_local = 0
        for idx in range(start, end):
            if eh_primo(elementos[idx]):
                count_local += 1
        return count_local

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        resultados = list(executor.map(contar_no_chunk, range(num_threads)))
    return sum(resultados)

# Testar a contagem de primos
N = 100000
t_ini = time.time()
primos = contar_primos_paralelo(N, num_threads=4)
print(f"Primos ate {N}: {primos} em {(time.time() - t_ini)*1000:.2f} ms")


### Grupo 4: Árvores de Prefixo (Trie IPv4 e LPM)
Implementação de árvore de prefixos e mecanismo de Longest Prefix Match (LPM) para roteamento IP.

In [ ]:
class TrieNodeIP:
    def __init__(self):
        self.children = {} # Inicializa filhos
        self.is_prefix = False
        self.prefix_str = None

class TrieIP:
    def __init__(self):
        self.root = TrieNodeIP()

    def _ip_to_bits(self, ip, cidr):
        parts = [int(p) for p in ip.split('.')]
        bits = "".join(f"{p:08b}" for p in parts)
        return bits[:cidr]

    def inserir(self, ip, cidr):
        # Insere prefixo na Trie
        bits = self._ip_to_bits(ip, cidr)
        curr = self.root
        for b in bits:
            if b not in curr.children:
                curr.children[b] = TrieNodeIP()
            curr = curr.children[b]
        curr.is_prefix = True
        curr.prefix_str = f"{ip}/{cidr}"

    def longest_prefix_match(self, ip):
        # Busca melhor correspondencia LPM
        bits = self._ip_to_bits(ip, 32)
        curr = self.root
        best_match = None
        for b in bits:
            if b in curr.children:
                curr = curr.children[b]
                if curr.is_prefix:
                    best_match = curr.prefix_str
            else:
                break
        return best_match

# Testando o LPM
trie_ip = TrieIP()
trie_ip.inserir("192.168.0.0", 16)
trie_ip.inserir("192.168.1.0", 24)

print("Busca 192.168.1.10:", trie_ip.longest_prefix_match("192.168.1.10"))
print("Busca 192.168.2.100:", trie_ip.longest_prefix_match("192.168.2.100"))

# Parte 4: Heaps, Tries, Grafos e Conectividade de Rede (TP4)


### Grupo 1: Heap Binária e Fila de Prioridades
Implementações de Heap Mínima, Heap Máxima e simulação de pouso de aviões.

In [ ]:
class MinHeap:
    def __init__(self):
        self.heap = []

    def push(self, val):
        self.heap.append(val)
        self._up(len(self.heap)-1)

    def pop(self):
        if not self.heap: return None
        if len(self.heap) == 1: return self.heap.pop()
        res = self.heap[0]
        self.heap[0] = self.heap.pop()
        self._down(0)
        return res

    def _up(self, idx):
        while idx > 0:
            p = (idx - 1) // 2
            if self.heap[idx] < self.heap[p]:
                self.heap[idx], self.heap[p] = self.heap[p], self.heap[idx]
                idx = p
            else: break

    def _down(self, idx):
        n = len(self.heap)
        while 2 * idx + 1 < n:
            c = 2 * idx + 1
            if c + 1 < n and self.heap[c+1] < self.heap[c]: c += 1
            if self.heap[idx] > self.heap[c]:
                self.heap[idx], self.heap[c] = self.heap[c], self.heap[idx]
                idx = c
            else: break

def build_heap(arr):
    # Converte vetor em heap
    h = MinHeap()
    for x in arr: h.push(x)
    return h.heap

print("Build heap de [7, 5, 9, 1]:", build_heap([7, 5, 9, 1]))

#### Fila de Prioridade Pousos de Aviões
Fila de pouso priorizada por combustível (menor combustível pousa primeiro) e desempate por tempo.

In [ ]:
import heapq

class Aviao:
    def __init__(self, id_av, tempo_sol, combustivel):
        self.id = id_av
        self.tempo_sol = tempo_sol
        self.combustivel = combustivel

    def __lt__(self, other):
        # Prioridade por combustivel descrescente/crescente
        if self.combustivel == other.combustivel:
            return self.tempo_sol < other.tempo_sol
        return self.combustivel < other.combustivel

def simula_pousos(avioes):
    # Determina ordem de pousos
    heap = []
    for av in avioes:
        heapq.heappush(heap, av)

    sem_combustivel = 0
    tempo_atual = 0
    while heap:
        av = heapq.heappop(heap)
        if tempo_atual > av.tempo_sol + av.combustivel:
            sem_combustivel += 1
        # Simula pouso leva 1 tempo
        tempo_atual += 1
        print(f"Aviao {av.id} pousou no tempo {tempo_atual}")
    print(f"Qtd avioes sem combustivel: {sem_combustivel}")

### Grupo 2: Estrutura Trie de Palavras e Autocomplete

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_word = False
        self.count = 0  # Palavras com o prefixo

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def inserir(self, word):
        # Insere palavra na Trie
        curr = self.root
        curr.count += 1
        for char in word:
            if char not in curr.children:
                curr.children[char] = TrieNode()
            curr = curr.children[char]
            curr.count += 1
        curr.is_word = True

    def autocomplete(self, prefix):
        # Autocomplete de palavras
        curr = self.root
        for char in prefix:
            if char not in curr.children: return []
            curr = curr.children[char]
        res = []
        self._dfs_words(curr, prefix, res)
        return res

    def _dfs_words(self, node, prefix, res):
        if node.is_word: res.append(prefix)
        for char, child in node.children.items():
            self._dfs_words(child, prefix + char, res)

t = Trie()
t.inserir("casa")
t.inserir("carro")
t.inserir("caminhao")
print("Autocomplete 'ca':", t.autocomplete("ca"))

### Grupo 3: Grafos e Busca BFS/DFS

In [ ]:
from collections import deque

class Grafo:
    def __init__(self):
        self.adj = {}

    def add_aresta(self, u, v):
        if u not in self.adj: self.adj[u] = []
        if v not in self.adj: self.adj[v] = []
        self.adj[u].append(v)
        self.adj[v].append(u)

    def bfs(self, start):
        # Busca em largura
        visited = set([start])
        queue = deque([start])
        order = []
        while queue:
            curr = queue.popleft()
            order.append(curr)
            for viz in self.adj.get(curr, []):
                if viz not in visited:
                    visited.add(viz)
                    queue.append(viz)
        return order

### Grupo 4 & 5: Sockets TCP/UDP e Diagnósticos de Rede
Implementamos servidores executados localmente em threads de background de forma funcional no Colab.

In [ ]:
import socket
import threading

def rodar_servidor_tcp():
    # Servidor simples TCP
    serv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    serv.bind(('127.0.0.1', 9991))
    serv.listen(1)
    conn, addr = serv.accept()
    msg = conn.recv(1024).decode()
    conn.send(f"ECO: {msg}".encode())
    conn.close()
    serv.close()

# Inicializa o servidor numa thread
t = threading.Thread(target=rodar_servidor_tcp)
t.start()

# Envia dados com o cliente
cli = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
cli.connect(('127.0.0.1', 9991))
cli.send(b"Ola Servidor!")
print("Resposta do servidor:", cli.recv(1024).decode())
cli.close()

# Parte 5: Grafos Avançados, Heurísticas e Segurança de Rede (TP5)


### Grupo 1: Algoritmos de Grafos (Dijkstra e Kruskal MST)
Implementação de caminhos mínimos e árvore geradora mínima.

In [ ]:
import heapq

def dijkstra(grafo, orig):
    # Calcula caminhos mínimos
    dist = {v: float('inf') for v in grafo}
    dist[orig] = 0
    pq = [(0, orig)]

    while pq:
        d_u, u = heapq.heappop(pq)
        if d_u > dist[u]: continue
        for v, peso in grafo.get(u, []):
            if dist[u] + peso < dist[v]:
                dist[v] = dist[u] + peso
                heapq.heappush(pq, (dist[v], v))
    return dist

grafo_ex = {
    'A': [('B', 2), ('C', 4)],
    'B': [('C', 1)],
    'C': []
}
print("Dijkstra de A:", dijkstra(grafo_ex, 'A'))

### Grupo 2: Heurísticas (Mochila e Caixeiro Viajante)
Estratégia gulosa para a mochila fracionária e vizinho mais próximo no TSP.

In [ ]:
def mochila_gulosa(itens, cap):
    # Ordena por valor peso
    itens.sort(key=lambda x: x[0]/x[1], reverse=True)
    val_tot = 0.0
    for val, peso in itens:
        if cap >= peso:
            cap -= peso
            val_tot += val
        else:
            val_tot += val * (cap / peso)
            break
    return val_tot

print("Valor mochila:", mochila_gulosa([(60, 10), (100, 20), (120, 30)], 50))

### Grupo 3: sockets com TLS (Simulação em Thread local)
Demonstra a comunicação criptografada utilizando o módulo `ssl` nativo do Python.

In [ ]:
import ssl
import socket
import threading

# Simulacao conceitual de TLS local
def rodar_tls_server():
    # Configura contexto TLS
    ctx = ssl.create_default_context(ssl.Purpose.CLIENT_AUTH)
    # (Necessitaria de certificados autoassinados no disco para carregar)
    # ctx.load_cert_chain(certfile="cert.pem", keyfile="key.pem")
    pass

### Grupo 4: Scanner de Portas e Leitores ARP usando Scapy

In [ ]:
def scanner_portas(ip, p_ini, p_fim):
    # Estabelece conexao TCP para varrer portas
    portas_abertas = []
    for port in range(p_ini, p_fim + 1):
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(0.1)
        res = s.connect_ex((ip, port))
        if res == 0: portas_abertas.append(port)
        s.close()
    return portas_abertas

print("Portas locais abertas:", scanner_portas('127.0.0.1', 8880, 8890))

### Grupo 5: Consulta DNS em Python
Resolve endereços DNS usando a biblioteca `socket` do Python.

In [ ]:
import time
import socket

def consulta_dns(dominio):
    # Consulta IP do dominio
    t_ini = time.time()
    try:
        ip = socket.gethostbyname(dominio)
        t_gasto = time.time() - t_ini
        print(f"Dominio: {dominio} | IP: {ip} | Tempo: {t_gasto*1000:.2f} ms")
    except socket.gaierror:
        print(f"Nao foi possivel resolver o dominio {dominio}")

consulta_dns("google.com")